> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 2 · Notebook 04 — Estimation, inference and regression

**Sessions:** S4 (Estimation, inference & regression) · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Quantify how uncertain a mean return (and a Sharpe ratio) really is.
2. Use the block bootstrap for dependent data.
3. Estimate CAPM betas with robust (HAC) standard errors.
4. See multiple testing create false discoveries.

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

## 1. How much data does a mean return need?

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
r = rets["SPY"]
t_stat = r.mean() / (r.std() / np.sqrt(len(r)))
t_stat = p.check("t-stat of SPY's mean return", t_stat, p.tstat_mean(r))
years = len(r) / 252
print(f"t = {t_stat:.2f};  annual Sharpe × sqrt(years) = {r.mean() / r.std() * np.sqrt(252) * np.sqrt(years):.2f}")

In [ ]:
years_05, years_10 = (2 / 0.5) ** 2, (2 / 1.0) ** 2
years_05 = p.check("years needed (Sharpe 0.5)", years_05, p.years_needed(0.5))
years_10 = p.check("years needed (Sharpe 1.0)", years_10, p.years_needed(1.0))

## 2. Confidence intervals: normal theory vs block bootstrap

In [ ]:
se = r.std() / np.sqrt(len(r))
normal_ci = (r.mean() - 1.96 * se, r.mean() + 1.96 * se)
boot_ci = p.block_bootstrap_ci(r, block=20)
print("95% CI for the mean daily return, annualized (%):")
print(f"  normal theory:   {normal_ci[0]*252:.2%} to {normal_ci[1]*252:.2%}")
print(f"  block bootstrap: {boot_ci[0]*252:.2%} to {boot_ci[1]*252:.2%}")

## 3. CAPM regressions with HAC standard errors

In [ ]:
import statsmodels.api as sm
rows = {}
for t in rets.columns.drop("SPY"):
    X = sm.add_constant(rets["SPY"])
    plain = sm.OLS(rets[t], X).fit()
    hac = sm.OLS(rets[t], X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    rows[t] = {"beta": hac.params["SPY"], "alpha (ann.)": hac.params["const"] * 252,
               "alpha t (plain)": plain.tvalues["const"], "alpha t (HAC)": hac.tvalues["const"], "R²": hac.rsquared}
capm = pd.DataFrame(rows).T
capm

## 4. Multiple testing

200 "strategies" that trade on random coin flips. None has any real edge.

In [ ]:
from scipy import stats
rng = np.random.default_rng(1)
pvals = []
for _ in range(200):
    signal = rng.choice([-1, 1], size=len(r))              # random long/short every day
    strat = signal * r.to_numpy()
    pvals.append(stats.ttest_1samp(strat, 0).pvalue)
pvals = np.array(pvals)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
n_sig = int((pvals < 0.05).sum())
n_sig_fdr = int((stats.false_discovery_control(pvals, method="bh") < 0.05).sum())
n_sig, n_sig_fdr = p.check("significant before / after FDR", (n_sig, n_sig_fdr), p.fdr_counts(pvals))
print(f"'Significant' strategies: {n_sig} of 200 before correction, {n_sig_fdr} after BH-FDR")

## Questions
1. With 10+ years of data, can you tell whether SPY's true Sharpe is 0.3 or 0.6? Use the confidence interval.
2. Why do HAC t-statistics differ from plain OLS ones? Which alphas survive?
3. You tested 200 ideas and 9 "worked". What should you report, and what should you do next?